Install Required Libraries

In [1]:
!pip install transformers
!pip install datasets
!pip install torch
!pip install -qU huggingface_hub
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

Hugging Face Login

In [ ]:
from huggingface_hub import login
login("your_token")

Install Merge Kit

In [4]:
!git clone https://github.com/arcee-ai/mergekit.git
!cd mergekit && pip install -qqq -e . --progress-bar off

Cloning into 'mergekit'...
remote: Enumerating objects: 3031, done.
remote: Counting objects: 100% (1230/1230), done.
remote: Compressing objects: 100% (361/361), done.
remote: Total 3031 (delta 1064), reused 869 (delta 869), pack-reused 1801 (from 3)
Receiving objects: 100% (3031/3031), 1.02 MiB | 11.16 MiB/s, done.
Resolving deltas: 100% (2081/2081), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mergekit (pyproject.toml) ... done


### SFT + RESTA

In [ ]:
yaml_config = """
models:
  - model: your_model_id
    parameters:
      weight: 1.0

  - model: your_model_id
    parameters:
      weight: 1.0
      lambda: 0.5

merge_method: task_arithmetic
base_model: google/gemma-3-1b-pt
parameters:
  int8_mask: true
dtype: bfloat16
"""

with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

Merging Models

In [6]:
!mergekit-yaml config.yaml merge --copy-tokenizer --cuda --low-cpu-memory

2025-04-09 13:22:13.815916: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744204934.185501    1174 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744204934.281810    1174 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-09 13:22:15.048210: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 874/874 [00:00<00:00, 5.66MB/s]
config.json: 100% 880/880 [00:00<00:00, 4.84MB/s]
config.json: 100%

Push Model to Hub

In [ ]:
from huggingface_hub import HfApi

MODEL_NAME = "your_model_id"

api = HfApi(token="your_token")

api.create_repo(
    repo_id=f"your_user_id/{MODEL_NAME}",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    repo_id=f"your_user_id/{MODEL_NAME}",
    folder_path="merge",
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/devichand/sft_resta/commit/493a11c989de8a92630b645c947fad61db84a0c7', commit_message='Upload folder using huggingface_hub', commit_description='', oid='493a11c989de8a92630b645c947fad61db84a0c7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devichand/sft_resta', endpoint='https://huggingface.co', repo_type='model', repo_id='devichand/sft_resta'), pr_revision=None, pr_num=None)

### PEFT + RESTA

In [ ]:
yaml_config = """
models:
  - model: your_model_id
    parameters:
      weight: 1.0

  - model: your_model_id
    parameters:
      weight: 1.0
      lambda: 0.5

merge_method: task_arithmetic
base_model: google/gemma-3-1b-pt
parameters:
  int8_mask: true
dtype: bfloat16
"""

with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

Merging Models

In [10]:
!mergekit-yaml config.yaml merge --copy-tokenizer --cuda --low-cpu-memory

2025-04-09 13:32:25.379033: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744205545.409372    3805 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744205545.419706    3805 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-09 13:32:25.457279: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 874/874 [00:00<00:00, 5.44MB/s]
config.json: 100% 874/874 [00:00<00:00, 5.31MB/s]
Warmup loader cac

Push Model to Hub

In [ ]:
from huggingface_hub import HfApi

MODEL_NAME = "your_model_id"

api = HfApi(token="your_token")

api.create_repo(
    repo_id=f"your_user_id/{MODEL_NAME}",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    repo_id=f"your_user_id/{MODEL_NAME}",
    folder_path="merge",
)

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/devichand/peft_resta/commit/e70dae2caf7203afc83b6fb8d3ddcd3e55afc1fe', commit_message='Upload folder using huggingface_hub', commit_description='', oid='e70dae2caf7203afc83b6fb8d3ddcd3e55afc1fe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devichand/peft_resta', endpoint='https://huggingface.co', repo_type='model', repo_id='devichand/peft_resta'), pr_revision=None, pr_num=None)

### SFT+DARE+RESTA

In [ ]:
yaml_config = """
models:
  - model: your_model_id
    parameters:
      weight: 1.0

  - model: your_model_id
    parameters:
      weight: 1.0
      lambda: 0.5

merge_method: task_arithmetic
base_model: google/gemma-3-1b-pt
parameters:
  int8_mask: true
dtype: bfloat16
"""

with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

Merging Models

In [13]:
!mergekit-yaml config.yaml merge --copy-tokenizer --cuda --low-cpu-memory

2025-04-09 13:40:37.919036: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744206037.953843    5895 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744206037.963953    5895 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-09 13:40:38.007159: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 875/875 [00:00<00:00, 4.73MB/s]
Warmup loader cache:   0% 0/3 [00:00<?, ?it/s]
Fetching 3 files: 10

Push model to Hub

In [ ]:
from huggingface_hub import HfApi

MODEL_NAME = "your_model_id"

api = HfApi(token="your_token")

api.create_repo(
    repo_id=f"your_user_id/{MODEL_NAME}",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    repo_id=f"your_user_id/{MODEL_NAME}",
    folder_path="merge",
)

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/devichand/sft_dare_resta/commit/633ed4b9b1a159f70c7c699a22d83a038cb54609', commit_message='Upload folder using huggingface_hub', commit_description='', oid='633ed4b9b1a159f70c7c699a22d83a038cb54609', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devichand/sft_dare_resta', endpoint='https://huggingface.co', repo_type='model', repo_id='devichand/sft_dare_resta'), pr_revision=None, pr_num=None)

### PEFT+DARE+RESTA

In [ ]:
yaml_config = """
models:
  - model: your_model_id
    parameters:
      weight: 1.0

  - model: your_model_id
    parameters:
      weight: 1.0
      lambda: 0.5

merge_method: task_arithmetic
base_model: google/gemma-3-1b-pt
parameters:
  int8_mask: true
dtype: bfloat16
"""

with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

Merging Models

In [16]:
!mergekit-yaml config.yaml merge --copy-tokenizer --cuda --low-cpu-memory

2025-04-09 13:47:37.395205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744206457.424922    7661 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744206457.439533    7661 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-09 13:47:37.501373: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 875/875 [00:00<00:00, 4.38MB/s]
Warmup loader cache:   0% 0/3 [00:00<?, ?it/s]
Fetching 7 files:   

Push model to Hub

In [ ]:
from huggingface_hub import HfApi

MODEL_NAME = "your_model_id"

api = HfApi(token="your_token")

api.create_repo(
    repo_id=f"your_user_id/{MODEL_NAME}",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    repo_id=f"your_user_id/{MODEL_NAME}",
    folder_path="merge",
)

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/devichand/peft_dare_resta/commit/3718505f92e5dccb4fc06dba5c4ddf3eaf60a40c', commit_message='Upload folder using huggingface_hub', commit_description='', oid='3718505f92e5dccb4fc06dba5c4ddf3eaf60a40c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devichand/peft_dare_resta', endpoint='https://huggingface.co', repo_type='model', repo_id='devichand/peft_dare_resta'), pr_revision=None, pr_num=None)